In [0]:
from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import os
import mlflow
from pyspark.sql import functions as F
import warnings
from credit_risk.feature_engineering import feature_engineering
warnings.filterwarnings('ignore')

env = "dev"
catalog_name = f"mlops_{env}"
schema_name = "dbk_credit_analytics"

model_catalog_name = "mlops_prd"
model_schema_name = "dbk_credit_risk"
model_name = "credit_risk_model_custom"

pyfunc_model_name = f"{model_catalog_name}.{model_schema_name}.{model_name}"
model_uri = f"models:/{pyfunc_model_name}@latest-model"

original_features = [
    "term",
    "earliest_cr_line",
    "issue_d",
    "funded_amnt",
    "loan_status",
    "int_rate",
    "sub_grade",
    "annual_inc",
    "inq_last_6mths",
    "total_rev_hi_lim",
    "purpose",
    "dti",
    "home_ownership",
    "initial_list_status",
    "verification_status",
    "addr_state"
]

final_features = [
    "int_rate",
    "sub_grade",
    "annual_inc",
    "inq_last_6mths",
    "total_rev_hi_lim",
    "purpose",
    "dti",
    "home_ownership",
    "initial_list_status",
    "verification_status",
    "mths_since_earliest_cr_line",
    "addr_state"
]

In [0]:
columns_selected_final = ['id', 'member_id'] + final_features

df_raw = spark.table("bigquery_credit_analytics_catalog.credit_analytics.loan_data")

num_records = df_raw.count()
print(f"Number of records in the dataset: {num_records}")

df = feature_engineering(df_raw)

df["issue_d_date"] = pd.to_datetime(df["issue_d_date"])
df = df[df["issue_d_date"].dt.year == 2015]
df = df[columns_selected_final].reset_index(drop=True)

In [0]:
local_path = mlflow.artifacts.download_artifacts(model_uri)
wheel_dir = os.path.join(local_path, "code")

wheel_file = [
    f for f in os.listdir(wheel_dir)
    if f.endswith(".whl")
][0]

wheel_path = os.path.join(wheel_dir, wheel_file)
%pip install {wheel_path} --quiet

In [0]:
loaded_model = mlflow.pyfunc.load_model(model_uri)
predictions = loaded_model.predict(pd.DataFrame(df[final_features].astype(str)))

In [0]:
df_pred = pd.DataFrame(pd.concat([df, pd.DataFrame(predictions)], axis=1))
df_pred_spark = spark.createDataFrame(df_pred).withColumnRenamed("Probability of Default", "probability_default")

final_columns = ["id", "member_id", "loan_amnt", "funded_amnt", "issue_d", "purpose", "total_pymnt", "int_rate", "addr_state", "home_ownership", "emp_length", "annual_inc", "earliest_cr_line", "term"]

cond_j = [df_pred_spark.id == df_raw.id, df_pred_spark.member_id == df_raw.member_id]
df_final = df_pred_spark.join(df_raw, cond_j, how="inner") \
                        .select(
                            *[df_raw[c] for c in final_columns],
                            df_pred_spark["probability_default"]
                        )


In [0]:
df_final.write.mode("overwrite").saveAsTable(
    f"{catalog_name}.{schema_name}.operations_credit_risk"
)